# 01 — Schema & Semantic Profiling

**Objetivo:** perfilar columnas física y semánticamente, con una primera clasificación de riesgo temporal/leakage.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Qué columnas existen realmente en los CSV locales?
- ¿Qué tipos infiere DuckDB?
- ¿Cuál es el missingness y cardinalidad aproximada?
- ¿Qué columnas parecen IDs, fechas, dinero, texto, geografía o PII?
- ¿Qué campos parecen mutables o posteriores a `t0` y requieren auditoría?

Las etiquetas de leakage de este notebook son **hints**, no decisiones finales.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

In [ ]:
from argos.profiling.schema import profile_all_datasets
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]
column_profile = profile_all_datasets(
    FILES,
    sample_size=cfg["csv_sample_size"],
    example_rows=cfg["example_rows"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    column_profile,
    paths.reports / "profiling" / "column_profile.parquet",
)

column_profile.head(30)

In [ ]:
# Prioridad de revisión temporal: campos que NO deben entrar automáticamente a un modelo.
column_profile.query("leakage_risk_hint == 'high'")[[
    "dataset", "column", "semantic_type", "temporal_role_hint", "leakage_risk_hint"
]].sort_values(["dataset", "column"])

In [ ]:
# Missingness alto: no implica eliminar. Primero hay que entender si depende de tiempo/segmento.
column_profile.loc[column_profile["null_rate"].ge(0.50), [
    "dataset", "column", "semantic_type", "null_rate", "approx_distinct_count"
]].sort_values("null_rate", ascending=False).head(100)

## Criterio de salida
Debe existir un `column_profile.parquet` y una lista explícita de campos que requieran revisión temporal antes de cualquier feature engineering.